# E-Commerce Sales Analytics
**Author:** Kriti Jain  
**Dataset:** [E-Commerce Sales and Customer Analytics – Kaggle](https://www.kaggle.com/datasets/datascikhan/e-commerce-sales-and-customer-analytics)  
**Files used:** `ecommerce_sales_customer_analytics_150k.csv`, `customer_master.csv`, `order_items.csv`, `product_catalog.csv`

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully.')

## 2. Load Data

In [ ]:
orders = pd.read_csv('ecommerce_sales_customer_analytics_150k.csv', low_memory=False)
customers = pd.read_csv('customer_master.csv')
order_items = pd.read_csv('order_items.csv')
products = pd.read_csv('product_catalog.csv')

# Parse dates
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['year'] = orders['order_date'].dt.year
orders['month'] = orders['order_date'].dt.month
orders['month_name'] = orders['order_date'].dt.strftime('%b')
orders['year_month'] = orders['order_date'].dt.to_period('M')

print(f'Orders: {orders.shape}')
print(f'Customers: {customers.shape}')
print(f'Order Items: {order_items.shape}')
print(f'Products: {products.shape}')

## 3. Data Overview

In [ ]:
print('=== Orders Dataset ===' )
print(orders.dtypes)
print('\nMissing values:')
print(orders.isnull().sum()[orders.isnull().sum() > 0])

In [ ]:
print('=== Descriptive Statistics (Numeric) ===')
orders[['gross_sales','discount_amount','tax_amount','shipping_cost','net_sales','profit','profit_margin_percentage','customer_rating']].describe().round(2)

## 4. Key Business Metrics

In [ ]:
total_revenue  = orders['net_sales'].sum()
total_profit   = orders['profit'].sum()
total_orders   = orders['order_id'].nunique()
aov            = orders['net_sales'].mean()
avg_rating     = orders['customer_rating'].mean()
return_rate    = (orders['return_status'].notna() & (orders['return_status'] != '')).mean() * 100

print(f'Total Revenue  : ${total_revenue:,.2f}')
print(f'Total Profit   : ${total_profit:,.2f}')
print(f'Total Orders   : {total_orders:,}')
print(f'Avg Order Value: ${aov:,.2f}')
print(f'Avg Rating     : {avg_rating:.2f}')
print(f'Return Rate    : {return_rate:.2f}%')

## 5. Revenue Trend Over Time

In [ ]:
monthly_rev = (
    orders.groupby('year_month')['net_sales']
    .sum()
    .reset_index()
)
monthly_rev['year_month_str'] = monthly_rev['year_month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_rev['year_month_str'], monthly_rev['net_sales'] / 1e6,
        color='#3b82d4', linewidth=2, marker='o', markersize=3)
ax.set_title('Monthly Revenue Trend (2021 – 2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (USD Millions)')
tick_step = max(1, len(monthly_rev) // 12)
ax.set_xticks(range(0, len(monthly_rev), tick_step))
ax.set_xticklabels(monthly_rev['year_month_str'].iloc[::tick_step], rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
plt.tight_layout()
plt.savefig('chart_revenue_trend.png', bbox_inches='tight')
plt.show()
print('Chart saved: chart_revenue_trend.png')

## 6. Annual Revenue & Profit Comparison

In [ ]:
annual = orders.groupby('year')[['net_sales', 'profit']].sum().reset_index()

x = np.arange(len(annual))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w/2, annual['net_sales'] / 1e6, w, label='Revenue', color='#3b82d4')
bars2 = ax.bar(x + w/2, annual['profit']    / 1e6, w, label='Profit',  color='#7c5cd8')
ax.set_xticks(x)
ax.set_xticklabels(annual['year'].astype(str))
ax.set_title('Annual Revenue vs Profit', fontsize=14, fontweight='bold')
ax.set_ylabel('USD Millions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.1f}M'))
ax.legend()
plt.tight_layout()
plt.savefig('chart_annual_rev_profit.png', bbox_inches='tight')
plt.show()
print('Chart saved: chart_annual_rev_profit.png')

## 7. Sales by Region

In [ ]:
region_sales = orders.groupby('region')['net_sales'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('muted', len(region_sales))
region_sales.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Net Sales by Region', fontsize=14, fontweight='bold')
ax.set_xlabel('Net Sales (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('chart_sales_region.png', bbox_inches='tight')
plt.show()

## 8. Top 10 Product Categories by Revenue

In [ ]:
# Merge order_items with products
items_prod = order_items.merge(products[['product_id','product_category','product_subcategory','brand']], on='product_id', how='left')

cat_rev = items_prod.groupby('product_category')['net_sales'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
cat_rev.sort_values().plot(kind='barh', ax=ax, color=sns.color_palette('viridis', len(cat_rev)))
ax.set_title('Top 10 Product Categories by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Net Sales (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('chart_top_categories.png', bbox_inches='tight')
plt.show()

## 9. Sales Channel Distribution

In [ ]:
channel = orders.groupby('sales_channel')['net_sales'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie
axes[0].pie(channel.values, labels=channel.index, autopct='%1.1f%%',
            colors=sns.color_palette('pastel', len(channel)), startangle=140)
axes[0].set_title('Sales Channel Share', fontsize=13, fontweight='bold')

# Bar
channel.plot(kind='bar', ax=axes[1], color=sns.color_palette('muted', len(channel)))
axes[1].set_title('Revenue by Sales Channel', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Net Sales (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.0f}M'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('chart_sales_channel.png', bbox_inches='tight')
plt.show()

## 10. Customer Segment Analysis

In [ ]:
seg = orders.groupby('customer_segment').agg(
    orders_count=('order_id', 'count'),
    total_revenue=('net_sales', 'sum'),
    avg_order_value=('net_sales', 'mean'),
    avg_profit=('profit', 'mean'),
    avg_rating=('customer_rating', 'mean')
).round(2).reset_index()

print(seg.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('Set2', len(seg))
bars = ax.bar(seg['customer_segment'], seg['total_revenue'] / 1e6, color=colors)
ax.bar_label(bars, labels=[f'${v:.1f}M' for v in seg['total_revenue']/1e6], padding=4)
ax.set_title('Revenue by Customer Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Net Sales (USD Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}M'))
plt.tight_layout()
plt.savefig('chart_customer_segment.png', bbox_inches='tight')
plt.show()

## 11. Payment Method Analysis

In [ ]:
pay = orders.groupby('payment_method')['net_sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
pay.plot(kind='bar', ax=ax, color=sns.color_palette('coolwarm', len(pay)))
ax.set_title('Revenue by Payment Method', fontsize=14, fontweight='bold')
ax.set_ylabel('Net Sales (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('chart_payment_method.png', bbox_inches='tight')
plt.show()

## 12. Profit Margin Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(orders['profit_margin_percentage'].dropna(), bins=50,
        color='#3b82d4', edgecolor='white', alpha=0.85)
ax.axvline(orders['profit_margin_percentage'].median(), color='red',
           linestyle='--', label=f"Median: {orders['profit_margin_percentage'].median():.1f}%")
ax.set_title('Distribution of Profit Margin %', fontsize=14, fontweight='bold')
ax.set_xlabel('Profit Margin %')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('chart_profit_margin_dist.png', bbox_inches='tight')
plt.show()

## 13. Shipping Method vs Delivery Performance

In [ ]:
delivery = orders.groupby(['shipping_method', 'delivery_status']).size().unstack(fill_value=0)

delivery_pct = delivery.div(delivery.sum(axis=1), axis=0) * 100

delivery_pct.plot(kind='bar', stacked=True, figsize=(10, 5),
                  color=sns.color_palette('Set2', delivery_pct.shape[1]))
plt.title('Delivery Status by Shipping Method (%)', fontsize=14, fontweight='bold')
plt.ylabel('Percentage')
plt.xlabel('')
plt.xticks(rotation=20)
plt.legend(title='Delivery Status', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('chart_delivery_status.png', bbox_inches='tight')
plt.show()

## 14. Marketing Channel Effectiveness

In [ ]:
mkt = orders.groupby('marketing_channel').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('net_sales', 'sum'),
    avg_revenue_per_order=('net_sales', 'mean'),
    avg_rating=('customer_rating', 'mean')
).round(2).sort_values('total_revenue', ascending=False)

print(mkt.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
mkt['total_revenue'].sort_values().plot(kind='barh', ax=ax,
    color=sns.color_palette('Blues_d', len(mkt)))
ax.set_title('Revenue by Marketing Channel', fontsize=14, fontweight='bold')
ax.set_xlabel('Net Sales (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('chart_marketing_channel.png', bbox_inches='tight')
plt.show()

## 15. Customer Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Age histogram
axes[0].hist(orders['customer_age'].dropna(), bins=30, color='#7c5cd8',
             edgecolor='white', alpha=0.85)
axes[0].set_title('Customer Age Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')

# Gender split
gender_rev = orders.groupby('gender')['net_sales'].sum()
axes[1].pie(gender_rev.values, labels=gender_rev.index, autopct='%1.1f%%',
            colors=['#3b82d4', '#e85c8b'], startangle=90)
axes[1].set_title('Revenue by Gender', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('chart_demographics.png', bbox_inches='tight')
plt.show()

## 16. Top Brands by Revenue

In [ ]:
brand_rev = items_prod.groupby('brand')['net_sales'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
brand_rev.sort_values().plot(kind='barh', ax=ax,
    color=sns.color_palette('rocket', len(brand_rev)))
ax.set_title('Top 10 Brands by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Net Sales (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('chart_top_brands.png', bbox_inches='tight')
plt.show()

## 17. Correlation Heatmap

In [ ]:
num_cols = ['quantity','gross_sales','discount_amount','tax_amount',
            'shipping_cost','net_sales','profit','profit_margin_percentage',
            'customer_rating','customer_lifetime_value','customer_age']

corr = orders[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix – Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_correlation.png', bbox_inches='tight')
plt.show()

## 18. Order Status Breakdown

In [ ]:
status_counts = orders['order_status'].value_counts()

fig, ax = plt.subplots(figsize=(7, 5))
status_counts.plot(kind='bar', ax=ax,
    color=sns.color_palette('Set3', len(status_counts)))
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=20)
for bar in ax.patches:
    ax.annotate(f'{int(bar.get_height()):,}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('chart_order_status.png', bbox_inches='tight')
plt.show()

## 19. Customer Rating Analysis

In [ ]:
rating_seg = orders.groupby('customer_segment')['customer_rating'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
axes[0].hist(orders['customer_rating'].dropna(), bins=30,
             color='#3b82d4', edgecolor='white', alpha=0.85)
axes[0].axvline(orders['customer_rating'].mean(), color='red',
                linestyle='--', label=f"Mean: {orders['customer_rating'].mean():.2f}")
axes[0].set_title('Distribution of Customer Ratings', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].legend()

# By segment
rating_seg.plot(kind='bar', ax=axes[1], color=sns.color_palette('Set2', len(rating_seg)))
axes[1].set_title('Avg Rating by Customer Segment', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Rating')
axes[1].set_ylim(3, 4)
axes[1].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig('chart_ratings.png', bbox_inches='tight')
plt.show()

## 20. RFM-Based Customer Segmentation

In [ ]:
snapshot_date = orders['order_date'].max() + pd.Timedelta(days=1)

rfm = orders.groupby('customer_id').agg(
    Recency=('order_date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('order_id', 'count'),
    Monetary=('net_sales', 'sum')
).reset_index()

# Score 1-4 per dimension
rfm['R_Score'] = pd.qcut(rfm['Recency'],  4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  4, labels=[1, 2, 3, 4])
rfm['RFM_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['M_Score'].astype(int)

def rfm_label(score):
    if score >= 10: return 'Champions'
    elif score >= 7: return 'Loyal'
    elif score >= 5: return 'Potential'
    else: return 'At Risk'

rfm['Segment'] = rfm['RFM_Score'].apply(rfm_label)

seg_counts = rfm['Segment'].value_counts()
print(seg_counts)

fig, ax = plt.subplots(figsize=(8, 5))
seg_counts.plot(kind='bar', ax=ax,
    color=sns.color_palette('viridis', len(seg_counts)))
ax.set_title('RFM Customer Segments', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Customers')
ax.tick_params(axis='x', rotation=10)
for bar in ax.patches:
    ax.annotate(f'{int(bar.get_height()):,}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.savefig('chart_rfm_segments.png', bbox_inches='tight')
plt.show()

## 21. Summary of Key Insights

In [ ]:
print('='*60)
print('  E-Commerce Sales Analytics – Key Insights Summary')
print('='*60)
print(f'  Total Revenue          : ${total_revenue:>15,.2f}')
print(f'  Total Profit           : ${total_profit:>15,.2f}')
print(f'  Overall Profit Margin  : {total_profit/total_revenue*100:>14.2f}%')
print(f'  Total Orders           : {total_orders:>15,}')
print(f'  Average Order Value    : ${aov:>15,.2f}')
print(f'  Average Customer Rating: {avg_rating:>15.2f}')
print(f'  Top Region             : {region_sales.idxmax()}')
print(f'  Top Sales Channel      : {channel.idxmax()}')
print('='*60)